# 15_0. Refresh Tableau Topic Classification View

ODS 원천 `kic_data_ods.intellytics_voc.intellytics_display_online_voc`의 전체 행을 보존하고, `pred_topic`, `topic_group` 두 컬럼만 추가한 최종 Tableau view를 생성·검증합니다.

- 최종 view: `sandbox.t_online_voc_analysis.intellytics_display_online_voc`
- 원천 행 수와 최종 view 행 수는 항상 같아야 합니다.
- 분류 대상이 아닌 행 또는 아직 미분류 행은 라벨 컬럼이 `NULL`로 남습니다.


In [ ]:
import importlib
import sys

from pyspark.sql import functions as F

PROJECT_ROOT = "/Workspace/Users/jungryo.lee@lge.com/prj_TV_voc"
SRC_ROOT = f"{PROJECT_ROOT}/src"
if SRC_ROOT not in sys.path:
    sys.path.append(SRC_ROOT)

import common.config_loader as config_loader
import ml.final_output_view as final_output_view

importlib.reload(config_loader)
importlib.reload(final_output_view)

from common.config_loader import get_output_table, get_source_table, load_config
from ml.final_output_view import create_or_replace_final_classification_view

config = load_config(f"{PROJECT_ROOT}/config/settings_intellytics.yaml")

ODS_TABLE = get_source_table(config, "raw_ods_table")
FINAL_VIEW = get_source_table(config, "final_tableau_view")
FINAL_DETAIL_TABLE = get_output_table(config, "classification_detail_final")
TOPIC_GROUP_TABLE = get_output_table(config, "topic_group")

print({
    "ods_table": ODS_TABLE,
    "final_view": FINAL_VIEW,
    "final_detail_table": FINAL_DETAIL_TABLE,
    "topic_group_table": TOPIC_GROUP_TABLE,
})


In [ ]:
# 기존 대상이 legacy managed table인 경우에만 view로 교체합니다.
result = create_or_replace_final_classification_view(
    spark,
    config,
    replace_existing_table_with_view=True,
)
result


In [ ]:
final_df = spark.table(FINAL_VIEW)

display(
    final_df
    .groupBy("pred_topic", "topic_group")
    .agg(F.count("*").alias("row_cnt"))
    .orderBy(F.desc("row_cnt"))
)

display(final_df.limit(100))
